# Fraud model on `gold.ml_features`

Trains on the `train` split (fraudTrain.csv period), picks each model's decision threshold on the latest 20 % of
train, and reports PR-AUC, precision and recall on the `test` split, which is never used for fitting or tuning.
A random model's PR-AUC equals the test fraud rate: that is the bar to beat.

In [ ]:
from pathlib import Path

import pandas as pd
from pyspark.ml.classification import GBTClassifier, LogisticRegression
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from fraud_lakehouse.model import (
    LABEL,
    SEED,
    best_threshold,
    evaluate,
    pipeline,
    prepare,
    time_split,
    top_features,
    with_class_weight,
)

spark = (
    SparkSession.builder.appName("jupyter")
    .config("spark.driver.memory", "512m")
    .config("spark.driver.host", "jupyter")
    .getOrCreate()
)

In [ ]:
ml_features = spark.table("gold.ml_features")
fit, validation = time_split(prepare(ml_features.where("split = 'train'")), validation_fraction=0.2)
fit = with_class_weight(fit).cache()
validation = validation.cache()
test = prepare(ml_features.where("split = 'test'")).cache()

splits = []
for name, df in [("fit", fit), ("validation", validation), ("test", test)]:
    splits.append(f"{name} {df.count():,} rows ({df.agg(F.avg(LABEL)).first()[0]:.4%} fraud)")
snapshot = spark.sql(
    "SELECT snapshot_id FROM gold.ml_features.snapshots ORDER BY committed_at DESC"
)
data = f"Data: gold.ml_features snapshot {snapshot.first()[0]}; " + ", ".join(splits)
print(data)

In [ ]:
def train_and_score(classifier):
    model = pipeline(classifier).fit(fit)
    threshold = best_threshold(model.transform(validation))
    return model, {"threshold": threshold, **evaluate(model.transform(test), threshold)}


lr_model, lr = train_and_score(LogisticRegression(labelCol=LABEL, weightCol="weight", maxIter=100))
gbt_model, gbt = train_and_score(
    GBTClassifier(labelCol=LABEL, weightCol="weight", maxIter=50, maxDepth=5, seed=SEED)
)

In [ ]:
test_rate = test.agg(F.avg(LABEL)).first()[0]
results = pd.DataFrame(
    [
        {"model": "Random", "pr_auc": test_rate, "precision": test_rate},
        {"model": "LogisticRegression", **lr},
        {"model": "GBT", **gbt},
    ]
).set_index("model")
top = top_features(gbt_model, gbt_model.transform(test.limit(1)), 5)

print(results.round(4).to_string())
print("\nGBT top 5 features:")
for name, weight in top:
    print(f"  {name:22} {weight:.3f}")

In [ ]:
def cell(value):
    return "–" if value is None or pd.isna(value) else f"{value:.4f}"


lines = ["| Model | Threshold | PR-AUC | Precision | Recall |", "|---|---|---|---|---|"]
for model, row in results.iterrows():
    lines.append(
        f"| {model} | {cell(row.threshold)} | {cell(row.pr_auc)}"
        f" | {cell(row.precision)} | {cell(row.recall)} |"
    )
features = ", ".join(f"`{name}` ({weight:.3f})" for name, weight in top)
lines += ["", f"GBT top 5 features: {features}", "", data]
Path("out").mkdir(exist_ok=True)
Path("out/results.md").write_text("\n".join(lines) + "\n")
spark.stop()